# Agent Failure Intelligence GraphRAG

This notebook builds a GraphRAG system for AI agent failures.

We use:

- Neo4j for the knowledge graph
- OpenAI GPT-4.1-mini for reasoning
- LangGraph for workflow steps
- Cypher for graph search
- A dataset with AI agent incidents

The goal is to find failures, root causes, and useful fixes.

In [46]:
import os
import re

import pandas as pd
from dotenv import load_dotenv
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph
from langgraph.graph import StateGraph, END

In [47]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [48]:
df = pd.read_csv("data/ai_agent_observability_dataset.csv")

print(df.shape)
df.head()

(10000, 28)


,incident_id,timestamp,agent_name,agent_type,task,tool,llm_model,error,root_cause,fix,...,user_impact,region,incident_duration_minutes,retry_count,customer_tier,sla_breached,workflow_id,agent_version,failure_category,business_impact
0,INC-100000,2024-09-12 09:14:38,AnalyticsAgent_2,AnalyticsAgent,Data Extraction,DataWarehouse,llama-3.3-70b,Data Quality Issue,Missing Fields,Data Cleaning,...,Medium,APAC,99,2,Enterprise,True,WF-1098,v1.0,Data,Operational Disruption
1,INC-100001,2026-05-05 15:12:39,DocumentAgent_2,DocumentAgent,Document Processing,DocumentParser,mistral-large,Invalid Response,Poor Prompt Design,Prompt Optimization,...,Medium,APAC,63,1,Free,False,WF-1067,v1.0,Prompting,Medium
2,INC-100002,2024-07-19 03:34:48,FinanceAgent_4,FinanceAgent,Financial Forecasting,SQLDatabase,gemini-1.5-pro,Timeout,Rate Limit,Retry Logic,...,Medium,US-West,93,1,Professional,True,WF-1031,v1.0,Performance,Operational Disruption
3,INC-100003,2025-06-14 06:58:18,MonitoringAgent_1,MonitoringAgent,KPI Monitoring,MonitoringAPI,gpt-4o,Data Quality Issue,Missing Fields,Data Cleaning,...,High,US-West,1338,5,Free,True,WF-1094,v1.0,Data,Operational Disruption
4,INC-100004,2025-03-29 08:23:36,PlanningAgent_1,PlanningAgent,Market Analysis,LLM_API,gpt-4.1,Invalid Response,Poor Prompt Design,Prompt Optimization,...,Critical,US-West,9396,7,Free,True,WF-1010,v1.0,Prompting,Revenue Loss


In [49]:
len(df)

10000

In [50]:
df.columns

Index(['incident_id', 'timestamp', 'agent_name', 'agent_type', 'task', 'tool',
       'llm_model', 'error', 'root_cause', 'fix', 'severity', 'environment',
       'status', 'team', 'department', 'response_time_ms', 'token_usage',
       'cost_usd', 'user_impact', 'region', 'incident_duration_minutes',
       'retry_count', 'customer_tier', 'sla_breached', 'workflow_id',
       'agent_version', 'failure_category', 'business_impact'],
      dtype='str')

In [51]:
#
df["sla_breached"] = df["sla_breached"].astype(bool)
df["timestamp"] = df["timestamp"].astype(str)
df.head()

,incident_id,timestamp,agent_name,agent_type,task,tool,llm_model,error,root_cause,fix,...,user_impact,region,incident_duration_minutes,retry_count,customer_tier,sla_breached,workflow_id,agent_version,failure_category,business_impact
0,INC-100000,2024-09-12 09:14:38,AnalyticsAgent_2,AnalyticsAgent,Data Extraction,DataWarehouse,llama-3.3-70b,Data Quality Issue,Missing Fields,Data Cleaning,...,Medium,APAC,99,2,Enterprise,True,WF-1098,v1.0,Data,Operational Disruption
1,INC-100001,2026-05-05 15:12:39,DocumentAgent_2,DocumentAgent,Document Processing,DocumentParser,mistral-large,Invalid Response,Poor Prompt Design,Prompt Optimization,...,Medium,APAC,63,1,Free,False,WF-1067,v1.0,Prompting,Medium
2,INC-100002,2024-07-19 03:34:48,FinanceAgent_4,FinanceAgent,Financial Forecasting,SQLDatabase,gemini-1.5-pro,Timeout,Rate Limit,Retry Logic,...,Medium,US-West,93,1,Professional,True,WF-1031,v1.0,Performance,Operational Disruption
3,INC-100003,2025-06-14 06:58:18,MonitoringAgent_1,MonitoringAgent,KPI Monitoring,MonitoringAPI,gpt-4o,Data Quality Issue,Missing Fields,Data Cleaning,...,High,US-West,1338,5,Free,True,WF-1094,v1.0,Data,Operational Disruption
4,INC-100004,2025-03-29 08:23:36,PlanningAgent_1,PlanningAgent,Market Analysis,LLM_API,gpt-4.1,Invalid Response,Poor Prompt Design,Prompt Optimization,...,Critical,US-West,9396,7,Free,True,WF-1010,v1.0,Prompting,Revenue Loss


In [52]:
# Connect to Neo4j
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)
print("Connected to Neo4j")

Connected to Neo4j


In [53]:
# Clear old graph
graph.query("""
MATCH (n)
DETACH DELETE n
""")

[]

In [54]:
constraints = [
    """ CREATE CONSTRAINT incident_id IF NOT EXISTS FOR (i:Incident) REQUIRE i.id IS UNIQUE """,
    """ CREATE CONSTRAINT agent_name IF NOT EXISTS FOR (a:Agent) REQUIRE a.name IS UNIQUE """,
    """ CREATE CONSTRAINT workflow_id IF NOT EXISTS FOR (w:Workflow) REQUIRE w.id IS UNIQUE """,
]

for constraint in constraints:
    graph.query(constraint)
print("Constraints created")

Constraints created


## Build Knowledge Graph

In [55]:
# Build knowledge graph in batches
BATCH_SIZE = 500
records = df.to_dict("records")
query = """ UNWIND $rows AS row 
MERGE (incident:Incident {id: row.incident_id}) SET incident.timestamp = row.timestamp, incident.severity = row.severity, incident.environment = row.environment, incident.status = row.status, incident.response_time_ms = row.response_time_ms, incident.token_usage = row.token_usage, incident.cost_usd = row.cost_usd, incident.user_impact = row.user_impact, incident.incident_duration_minutes = row.incident_duration_minutes, incident.retry_count = row.retry_count, incident.customer_tier = row.customer_tier, incident.sla_breached = row.sla_breached, incident.failure_category = row.failure_category 
MERGE (agent:Agent {name: row.agent_name}) SET agent.type = row.agent_type, agent.version = row.agent_version 
MERGE (task:Task {name: row.task}) 
MERGE (tool:Tool {name: row.tool})
MERGE (model:Model {name: row.llm_model}) 
MERGE (error:Error {name: row.error}) 
MERGE (root:RootCause {name: row.root_cause}) 
MERGE (fix:Fix {name: row.fix}) 
MERGE (workflow:Workflow {id: row.workflow_id}) 
MERGE (team:Team {name: row.team}) 
MERGE (department:Department {name: row.department}) 
MERGE (impact:BusinessImpact {name: row.business_impact}) 
MERGE (region:Region {name: row.region}) 
MERGE (incident)-[:BELONGS_TO]->(agent) 
MERGE (incident)-[:PART_OF]->(workflow) 
MERGE (incident)-[:GENERATES]->(error) 
MERGE (incident)-[:IMPACTS]->(impact) 
MERGE (incident)-[:OCCURS_IN]->(region) 
MERGE (agent)-[:EXECUTES]->(task) 
MERGE (agent)-[:USES]->(tool) 
MERGE (agent)-[:RUNS_ON]->(model) 
MERGE (agent)-[:OWNED_BY]->(team) 
MERGE (team)-[:PART_OF]->(department) 
MERGE (error)-[:CAUSED_BY]->(root) 
MERGE (root)-[:RESOLVED_BY]->(fix) """

for i in range(0, len(records), BATCH_SIZE):
    batch = records[i : i + BATCH_SIZE]
    graph.query(query, params={"rows": batch})
    print(f"Processed {min(i + BATCH_SIZE, len(records))}/{len(records)}")
print("Knowledge graph created")

Processed 500/10000
Processed 1000/10000
Processed 1500/10000
Processed 2000/10000
Processed 2500/10000
Processed 3000/10000
Processed 3500/10000
Processed 4000/10000
Processed 4500/10000
Processed 5000/10000
Processed 5500/10000
Processed 6000/10000
Processed 6500/10000
Processed 7000/10000
Processed 7500/10000
Processed 8000/10000
Processed 8500/10000
Processed 9000/10000
Processed 9500/10000
Processed 10000/10000
Knowledge graph created


# Inspect Graph

In [56]:
# Check Graph size
graph.query("""
MATCH (n)
RETURN count(n) AS total_nodes
""")
graph.query("""
MATCH ()-[r]->()
RETURN count(r) AS total_relationships
""")

[{'total_relationships': 50227}]

In [57]:
# Refresh Schema
graph.refresh_schema()
print(graph.schema)

Node properties:
Agent {name: STRING, type: STRING, version: STRING}
Workflow {id: STRING}
Task {name: STRING}
Tool {name: STRING}
Model {name: STRING}
Error {name: STRING}
RootCause {name: STRING}
Fix {name: STRING}
Team {name: STRING}
BusinessImpact {name: STRING}
Incident {id: STRING, timestamp: STRING, severity: STRING, environment: STRING, status: STRING, sla_breached: BOOLEAN, response_time_ms: INTEGER, token_usage: INTEGER, cost_usd: FLOAT, user_impact: STRING, incident_duration_minutes: INTEGER, retry_count: INTEGER, customer_tier: STRING, failure_category: STRING}
Region {name: STRING}
Department {name: STRING}
Relationship properties:

The relationships:
(:Agent)-[:EXECUTES]->(:Task)
(:Agent)-[:USES]->(:Tool)
(:Agent)-[:RUNS_ON]->(:Model)
(:Agent)-[:OWNED_BY]->(:Team)
(:Error)-[:CAUSED_BY]->(:RootCause)
(:RootCause)-[:RESOLVED_BY]->(:Fix)
(:Team)-[:PART_OF]->(:Department)
(:Incident)-[:PART_OF]->(:Workflow)
(:Incident)-[:IMPACTS]->(:BusinessImpact)
(:Incident)-[:BELONGS_TO]->

## Initialize OpenAI LLM


In [58]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

## Create GraphCypherQAChain

In [59]:
GRAPH_SCHEMA = """ 
Nodes: 
Incident, 
Agent, 
Task,
Tool, 
Model, 
Error, 
RootCause, 
Fix, 
Workflow, 
Team, 
Department, 
BusinessImpact, 
Region 
Relationships: 
(Incident)-[:BELONGS_TO]->(Agent) 
(Incident)-[:PART_OF]->(Workflow) 
(Incident)-[:GENERATES]->(Error) 
(Incident)-[:IMPACTS]->(BusinessImpact) 
(Incident)-[:OCCURS_IN]->(Region) 
(Agent)-[:EXECUTES]->(Task) 
(Agent)-[:USES]->(Tool) 
(Agent)-[:RUNS_ON]->(Model) 
(Agent)-[:OWNED_BY]->(Team) 
(Team)-[:PART_OF]->(Department) 
(Error)-[:CAUSED_BY]->(RootCause) 
(RootCause)-[:RESOLVED_BY]->(Fix) 
Important properties: 
Incident.id 
Incident.severity 
Incident.environment 
Incident.status 
Incident.sla_breached 
Incident.customer_tier 
Incident.cost_usd 
Incident.token_usage 
Incident.response_time_ms 
Incident.failure_category 

Agent.name 
Agent.type 
Agent.version 

Workflow.id 
Model.name 
Error.name 
RootCause.name 
Fix.name 
Team.name 
Department.name 
Region.name 
BusinessImpact.name"""

In [60]:
def clean_cypher(text):
    text = text.strip()
    text = re.sub(r"```cypher", "", text)
    text = re.sub(r"```", "", text)
    return text.strip()

In [61]:
def generate_cypher(question):
    prompt = f""" 
    You are a Neo4j Cypher expert. Use only this graph schema: 
    {GRAPH_SCHEMA} 
    Rules: 
    - Return only Cypher. 
    - Do not explain.
    - Do not use markdown.
    - Use CONTAINS for agent groups like SupportAgent or ResearchAgent. 
    - Limit results to 10 unless the question asks for more. 
    Question: 
    {question} 
    """
    response = llm.invoke(prompt)
    cypher = clean_cypher(response.content)
    return cypher

In [62]:
def run_graph_query(question):
    cypher = generate_cypher(question)
    print("Generated Cypher:")
    print(cypher)
    results = graph.query(cypher)
    return results

In [63]:
def graph_rag(question):
    results = run_graph_query(question)
    prompt = f""" You are an AI agent failure analyst. Use the graph results to answer the question. 
    Question: 
    {question} 
    Graph results: 
    {results} 
    Write in simple English. Include: 1. Main finding 2. Root cause 3. Recommended action """
    response = llm.invoke(prompt)
    return response.content

In [64]:
graph.query("""
MATCH (i:Incident)-[:GENERATES]->(e:Error)
RETURN e.name AS error, count(*) AS incidents
ORDER BY incidents DESC
LIMIT 10
""")
graph.query("""
MATCH (i:Incident)-[:BELONGS_TO]->(a:Agent)
WHERE a.name CONTAINS 'SupportAgent'
MATCH (i)-[:GENERATES]->(e:Error)
RETURN e.name AS error, count(*) AS incidents
ORDER BY incidents DESC
LIMIT 10
""")
graph.query("""
MATCH (i:Incident)-[:PART_OF]->(w:Workflow)
RETURN w.id AS workflow, count(*) AS incidents
ORDER BY incidents DESC
LIMIT 10
""")

[{'workflow': 'WF-1020', 'incidents': 128},
 {'workflow': 'WF-1049', 'incidents': 125},
 {'workflow': 'WF-1063', 'incidents': 121},
 {'workflow': 'WF-1089', 'incidents': 121},
 {'workflow': 'WF-1046', 'incidents': 119},
 {'workflow': 'WF-1028', 'incidents': 118},
 {'workflow': 'WF-1090', 'incidents': 117},
 {'workflow': 'WF-1050', 'incidents': 115},
 {'workflow': 'WF-1060', 'incidents': 115},
 {'workflow': 'WF-1037', 'incidents': 114}]

In [65]:
print(graph_rag("Why does SupportAgent fail most often?"))
print(graph_rag("What root causes are linked to hallucination incidents?"))
print(graph_rag("Which workflows generate the most incidents?"))
print(graph_rag("What fixes resolve retrieval failures?"))

Generated Cypher:
MATCH (agent:Agent)-[:EXECUTES]->(task)<-[:EXECUTES]-(a:Agent)
WHERE agent.type CONTAINS 'SupportAgent'
WITH agent
MATCH (incident:Incident)-[:BELONGS_TO]->(agent)-[:EXECUTES]->(:Task)
MATCH (incident)-[:GENERATES]->(error:Error)
RETURN error.name AS RootCause, count(incident) AS Failures
ORDER BY Failures DESC
LIMIT 10
1. Main finding: SupportAgent fails most often because of a specific problem.  
2. Root cause: The biggest reason for failure is "Authentication Error," which happened 1,383 times.  
3. Recommended action: Fix the authentication process to reduce these errors and make SupportAgent work better.
Generated Cypher:
MATCH (i:Incident)-[:GENERATES]->(:Error)<-[:CAUSED_BY]-(rc:RootCause)
WHERE i.failure_category CONTAINS 'hallucination'
RETURN DISTINCT rc.name
LIMIT 10
1. Main finding: There is no information available about root causes linked to hallucination incidents.  
2. Root cause: Unknown, as no data or graph results were provided.  
3. Recommended act

In [66]:
print(graph_rag("Which LLM models are linked to critical incidents?"))
print(graph_rag("Which regions have the most SLA breaches?"))
print(graph_rag("Which teams have the highest operational risk?"))
print(graph_rag("What are the most common root causes in Production?"))

Generated Cypher:
MATCH (i:Incident)-[:BELONGS_TO]->(a:Agent)-[:RUNS_ON]->(m:Model)
WHERE i.severity = 'critical'
RETURN DISTINCT m.name
LIMIT 10
1. Main finding: There are no LLM models linked to critical incidents shown in the graph.  
2. Root cause: The graph does not contain any data or connections about LLM models and critical incidents.  
3. Recommended action: Collect and include data on LLM models and their incidents to analyze any possible links in the future.
Generated Cypher:
MATCH (i:Incident)
WHERE i.sla_breached = true
WITH i.OCCURS_IN AS region, count(i) AS breaches
RETURN region.name AS Region, breaches
ORDER BY breaches DESC
LIMIT 10


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `OCCURS_IN` does not exist in database `273224e8`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=8, offset=54>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 54, 'line': 3, 'column': 8}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (i:Incident)\nWHERE i.sla_breached = true\nWITH i.OCCURS_IN AS region, count(i) AS breaches\nRETURN region.name AS Region, breaches\nORDER BY breaches DESC\nLIMIT 10'


1. Main finding: The data does not show any specific regions with SLA breaches. Instead, all 6,566 breaches are grouped without region information.

2. Root cause: The root cause is missing or incomplete data about the regions where the breaches happened.

3. Recommended action: Collect and include region information in the data to identify which areas have the most SLA breaches. This will help target improvements effectively.
Generated Cypher:
MATCH (team:Team)<-[:OWNED_BY]-(agent:Agent)<-[:BELONGS_TO]-(incident:Incident)
WITH team, 
     sum(CASE WHEN incident.severity >= 4 THEN 1 ELSE 0 END) AS highSeverityCount,
     sum(incident.cost_usd) AS totalCost,
     sum(CASE WHEN incident.sla_breached = true THEN 1 ELSE 0 END) AS slaBreachedCount
WITH team, (highSeverityCount * 3 + totalCost * 0.001 + slaBreachedCount * 5) AS operationalRiskScore
RETURN team.name, operationalRiskScore
ORDER BY operationalRiskScore DESC
LIMIT 10
1. Main finding: The teams with the highest operational risk a

## LangGraph Agent Workflow

In [67]:
class GraphState(TypedDict):
    question: str
    cypher: str
    graph_results: str
    answer: str

In [68]:
def cypher_node(state):
    cypher = generate_cypher(state["question"])
    return {"cypher": cypher}

## Retrieve Graph Context

In [69]:
def retrieve_node(state):
    results = graph.query(state["cypher"])
    return {"graph_results": str(results)}

## Generate Final Answer

In [70]:
def answer_node(state):
    prompt = f""" You are an AI agent failure analyst. 
    Question: 
    {state["question"]} 
    Cypher: 
    {state["cypher"]} 
    Graph results: 
    {state["graph_results"]} 
    Write a simple answer. Include: - Main finding - Reason - Recommended action """
    response = llm.invoke(prompt)
    return {"answer": response.content}

# Build Workflow

In [71]:
workflow = StateGraph(GraphState)
workflow.add_node("generate_cypher", cypher_node)
workflow.add_node("retrieve_graph", retrieve_node)
workflow.add_node("generate_answer", answer_node)
workflow.set_entry_point("generate_cypher")
workflow.add_edge("generate_cypher", "retrieve_graph")
workflow.add_edge("retrieve_graph", "generate_answer")
workflow.add_edge("generate_answer", END)
app = workflow.compile()
print("LangGraph workflow ready")

LangGraph workflow ready


# Run GraphRAG

In [72]:
response = app.invoke(
    {"question": "Which agent types have the most critical incidents?"}
)

print(response["answer"])
response = app.invoke(
    {"question": "Recommend actions to reduce hallucination incidents."}
)

print(response["answer"])
response = app.invoke({"question": "Which workflows have the most SLA breaches?"})

print(response["answer"])

Main finding: There are no recorded critical incidents associated with any agent types in the current data.

Reason: The query returned an empty result set, indicating that either no incidents have been classified as critical or the data is missing or incomplete.

Recommended action: Verify the incident data for completeness and accuracy, ensure that severity levels are properly assigned, and update the dataset if necessary to enable meaningful analysis of critical incidents by agent type.
Main finding: There are no recorded fixes linked to hallucination failure incidents in the current data.

Reason: The graph database does not contain any resolved root causes or fixes specifically associated with hallucination-related failures.

Recommended action: Conduct a detailed investigation to identify root causes of hallucination incidents and document effective fixes. Meanwhile, implement general best practices such as improving training data quality, enhancing model validation, and incorpor